<a href="https://colab.research.google.com/github/ccfmartin/ccfmartin/blob/main/unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BraTS2020 Brain Tumor Segmentation

In [ ]:
!pip install opendatasets
!pip install pandas
!pip install h5py

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import copy
import os
import random
import time
import shutil
import zipfile
from math import atan2, cos, sin, sqrt, pi, log

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from numpy import linalg as LA
from torch import optim, nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset, random_split
from torch.utils.data.dataset import Dataset
from torchvision import transforms
from tqdm import tqdm

# 2. Model Architecture

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_op = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv_op(x)

In [ ]:
class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        down = self.conv(x)
        p = self.pool(down)

        return down, p

In [ ]:
class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels//2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x1, x2], 1)
        return self.conv(x)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.down_convolution_1 = DownSample(in_channels, 64)
        self.down_convolution_2 = DownSample(64, 128)
        self.down_convolution_3 = DownSample(128, 256)
        self.down_convolution_4 = DownSample(256, 512)

        self.bottle_neck = DoubleConv(512, 1024)

        self.up_convolution_1 = UpSample(1024, 512)
        self.up_convolution_2 = UpSample(512, 256)
        self.up_convolution_3 = UpSample(256, 128)
        self.up_convolution_4 = UpSample(128, 64)

        self.out = nn.Conv2d(in_channels=64, out_channels=num_classes, kernel_size=1)

    def forward(self, x):
        down_1, p1 = self.down_convolution_1(x)
        down_2, p2 = self.down_convolution_2(p1)
        down_3, p3 = self.down_convolution_3(p2)
        down_4, p4 = self.down_convolution_4(p3)

        b = self.bottle_neck(p4)

        up_1 = self.up_convolution_1(b, down_4)
        up_2 = self.up_convolution_2(up_1, down_3)
        up_3 = self.up_convolution_3(up_2, down_2)
        up_4 = self.up_convolution_4(up_3, down_1)

        out = self.out(up_4)
        return out

In [ ]:
def dice_score(pred, target, smooth=1e-5):
    # pred: logits from model (B, C, H, W)
    # target: class indices (B, H, W)

    pred = torch.argmax(pred, dim=1)  # Convert logits to class indices

    dice = 0.0
    num_classes = 3  # BraTS has 3 tumor classes

    for cls in range(num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice += (2.0 * intersection + smooth) / (union + smooth)

    return dice / num_classes

In [ ]:
def show_sample(image, mask, pred):
    image = image.cpu().numpy()
    mask = mask.cpu().numpy()

    pred = torch.softmax(pred, dim=0)
    pred = torch.argmax(pred, dim=0)

    plt.figure(figsize=(10,3))

    plt.subplot(1,3,1)
    plt.title("Image")
    plt.imshow(image[0], cmap="gray")

    plt.subplot(1,3,2)
    plt.title("Ground Truth")
    plt.imshow(mask[0], cmap="gray")

    plt.subplot(1,3,3)
    plt.title("Prediction")
    plt.imshow(pred[0], cmap="gray")

    plt.show()

In [ ]:
input_image = torch.rand((1,3,512,512))
model = UNet(3,10)
output = model(input_image)
print(output.size())
# You should get torch.Size([1, 10, 512, 512]) as a result

# Data

In [ ]:
# Download BraTS2020 Dataset

import opendatasets as od
import pandas

od.download("https://www.kaggle.com/datasets/awsaf49/brats2020-training-data")

In [ ]:
dataset_path = "/content/brats2020-training-data/BraTS2020_training_data/content/data"
file = pandas.read_csv('/content/brats2020-training-data/BraTS20 Training Metadata.csv')
print(file)

In [ ]:
files = os.listdir(dataset_path)
print(files[:5])
sample_file = os.path.join(dataset_path, files[0])
with h5py.File(sample_file, 'r') as f:
  print(list(f.keys()))

In [ ]:
with h5py.File(sample_file, 'r') as f:
  image = f['image'][:]
  mask = f['mask'][:]

print(image.shape)
print(mask.shape)

In [ ]:
class BraTSDataset(Dataset):

    def __init__(self, file_list, augment=False):
        self.file_list = file_list
        self.augment = augment

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):

        file_path = self.file_list[idx]

        with h5py.File(file_path, 'r') as f:

            image = f['image'][:]
            mask = f['mask'][:]

        # normalize image range
        image = image / 255.0

        # data augmentation
        if self.augment:

            # horizontal flip
            if random.random() > 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()

            # vertical flip
            if random.random() > 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()

        # convert image to tensor
        image = torch.tensor(image, dtype=torch.float32)

        # HWC -> CHW
        image = image.permute(2, 0, 1)

        # standardization
        image = (image - image.mean()) / (image.std() + 1e-8)

        # convert mask
        mask = torch.tensor(mask, dtype=torch.long)  # Use long for class indices

        # one-hot -> class labels (H, W, C) -> (H, W)
        if len(mask.shape) == 3:
            mask = torch.argmax(mask, dim=-1)  # Now shape (H, W)
        else:
            # If already class indices, ensure correct dtype
            mask = mask.long()

        # Add channel dimension to match model output? No, keep as (H, W)
        # The loss function expects (B, H, W) for target

        return image, mask

In [ ]:
file_paths = [
    os.path.join(dataset_path, f)
    for f in os.listdir(dataset_path)
    if f.endswith('.h5')
]

In [ ]:
# dataset = BraTSDataset(file_paths)
# training dataset uses augmentation
train_dataset_full = BraTSDataset(
    file_paths,
    augment=True
)

# validation/test datasets do NOT use augmentation
eval_dataset_full = BraTSDataset(
    file_paths,
    augment=False
)

# randomly choose 2000 samples
subset_indices = np.random.choice(
    len(file_paths),
    2000,
    replace=False
)

np.random.shuffle(subset_indices)

# split indices
train_size = int(0.7 * len(subset_indices))
val_size = int(0.15 * len(subset_indices))
test_size = len(subset_indices) - train_size - val_size

train_indices = subset_indices[:train_size]

val_indices = subset_indices[
    train_size : train_size + val_size
]

test_indices = subset_indices[
    train_size + val_size :
]

# create subsets
train_dataset = Subset(
    train_dataset_full,
    train_indices
)

val_dataset = Subset(
    eval_dataset_full,
    val_indices
)

test_dataset = Subset(
    eval_dataset_full,
    test_indices
)

# dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
model = UNet(
    in_channels=4,
    num_classes=3
)

In [ ]:
# 1. Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=2,
    factor=0.5
)

# Model Training

In [ ]:
scaler = GradScaler()

num_epochs = 10
train_loss_history = []
train_dice_history = []

val_loss_history = []
val_dice_history = []

best_dice = 0.0

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    dice_total = 0.0
    num_batches = 0
    # Move data to GPU inside the training loop
    for images, masks in train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type):
          outputs = model(images)
          loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)

        scaler.update()

        # loss tracking
        running_loss += loss.item()

        # dice tracking
        dice = dice_score(outputs, masks)
        dice_total += dice.item()
        num_batches += 1

    # VALIDATION

    model.eval()

    val_loss = 0.0
    val_dice_total = 0.0
    val_batches = 0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            with torch.autocast(device_type=device.type):
              outputs = model(images)
              loss = criterion(outputs, masks)

            val_loss += loss.item()
            dice = dice_score(outputs, masks)

            val_dice_total += dice.item()
            val_batches += 1



    avg_train_loss = running_loss / num_batches
    avg_train_dice = dice_total / num_batches

    avg_val_loss = val_loss / val_batches
    avg_val_dice = val_dice_total / val_batches

    scheduler.step(avg_val_dice)

    train_loss_history.append(avg_train_loss)
    train_dice_history.append(avg_train_dice)

    val_loss_history.append(avg_val_loss)
    val_dice_history.append(avg_val_dice)

    epoch_time = time.time() - start_time

    print(
    f"Time: {epoch_time:.2f}s | "
    f"Epoch {epoch+1} | "
    f"Train Loss: {avg_train_loss:.4f} | "
    f"Train Dice: {avg_train_dice:.4f} | "
    f"Val Loss: {avg_val_loss:.4f} | "
    f"Val Dice: {avg_val_dice:.4f}"
    )

    # SAVE CHECKPOINT (good)
    if avg_val_dice > best_dice:

      best_dice = avg_val_dice

      torch.save({
          'epoch': epoch,
          'best_dice': avg_val_dice,
          'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'scheduler_state_dict': scheduler.state_dict(),
          'scaler_state_dict': scaler.state_dict(),
      }, 'best_model.pth')

      print("Best model saved.")

In [ ]:
print(masks.unique())

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(train_loss_history, label='Train Loss')
plt.plot(val_loss_history, label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()
plt.title('Training vs Validation Loss')

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(train_dice_history, label='Train Dice')
plt.plot(val_dice_history, label='Validation Dice')

plt.xlabel('Epoch')
plt.ylabel('Dice')

plt.legend()
plt.title('Training vs Validation Dice')

plt.show()

In [ ]:
model.eval()

# Grab a batch
images, masks = next(iter(val_loader))
images, masks = images.to(device), masks.to(device)

# Forward pass
with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

idx = 0
plt.figure(figsize=(18, 6))

# Show all 4 input modalities
modalities = ['T1', 'T1ce', 'T2', 'FLAIR']
for i in range(4):
    plt.subplot(2, 4, i+1)
    img_display = images[idx][i].cpu()
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-8)
    plt.imshow(img_display, cmap='gray')
    plt.title(modalities[i])
    plt.axis('off')

# Ground Truth
plt.subplot(2, 4, 5)
gt_display = masks[idx].cpu()
plt.imshow(gt_display, cmap='viridis', vmin=0, vmax=2)
plt.title("Ground Truth")
plt.axis('off')

# Prediction
plt.subplot(2, 4, 6)
plt.imshow(preds[idx].cpu(), cmap='viridis', vmin=0, vmax=2)
plt.title("Prediction")
plt.axis('off')


# Optional: Overlay prediction on FLAIR
plt.subplot(2, 4, 7)
flair = images[idx][3].cpu()
flair_norm = (flair - flair.min()) / (flair.max() - flair.min() + 1e-8)
plt.imshow(flair_norm, cmap='gray')
plt.imshow(preds[idx].cpu(), cmap='viridis', alpha=0.5, vmin=0, vmax=2)
plt.title("Prediction Overlay on FLAIR")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on full test set
model.eval()
test_dice_scores = []  # Store per-batch scores

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        dice = dice_score(outputs, masks)
        test_dice_scores.append(dice.item())

avg_test_dice = sum(test_dice_scores) / len(test_dice_scores)
print(f"Average Test Dice Score: {avg_test_dice:.4f}")
print(f"Per-batch Dice scores: {[f'{d:.4f}' for d in test_dice_scores]}")

In [ ]:
# Calculate Dice for this specific sample
with torch.no_grad():
    outputs = model(images)
    sample_dice = dice_score(outputs, masks).item()
    preds = torch.argmax(outputs, dim=1)

# Create figure with Dice score in title
plt.figure(figsize=(16, 8))

# Row 1: Input modalities
modalities = ['T1', 'T1ce', 'T2', 'FLAIR']
for i in range(4):
    plt.subplot(2, 4, i+1)
    img_display = images[idx][i].cpu()
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-8)
    plt.imshow(img_display, cmap='gray')
    plt.title(f'{modalities[i]}', fontsize=12)
    plt.axis('off')

# Row 2: Ground Truth, Prediction, Overlay, and Metrics
plt.subplot(2, 4, 5)
gt_display = masks[idx].cpu()
plt.imshow(gt_display, cmap='viridis', vmin=0, vmax=2)
plt.title('Ground Truth', fontsize=12)
plt.axis('off')

plt.subplot(2, 4, 6)
plt.imshow(preds[idx].cpu(), cmap='viridis', vmin=0, vmax=2)
plt.title(f'Prediction (Dice: {sample_dice:.3f})', fontsize=12)  # ← Dice in title
plt.axis('off')

# Overlay on FLAIR
plt.subplot(2, 4, 7)
flair = images[idx][3].cpu()
flair_norm = (flair - flair.min()) / (flair.max() - flair.min() + 1e-8)
plt.imshow(flair_norm, cmap='gray')
plt.imshow(preds[idx].cpu(), cmap='viridis', alpha=0.5, vmin=0, vmax=2)
plt.title('Prediction Overlay', fontsize=12)
plt.axis('off')

# Color Legend with Dice info
plt.subplot(2, 4, 8)
plt.axis('off')
legend_elements = [
    plt.Rectangle((0,0),1,1, facecolor='darkviolet', label='Class 0: Background'),
    plt.Rectangle((0,0),1,1, facecolor='lime', label='Class 1: Edema'),
    plt.Rectangle((0,0),1,1, facecolor='yellow', label='Class 2: Enhancing')
]
plt.legend(handles=legend_elements, loc='center', fontsize=10)
plt.title(f'Results\nDice Score: {sample_dice:.3f}\nAvg Test Dice: {avg_test_dice:.3f}', fontsize=12)

plt.suptitle('Brain Tumor Segmentation with Multi-Modal MRI', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def per_class_dice(pred_logits, target, smooth=1e-5):
    """Calculate Dice score for each class separately"""
    pred = torch.argmax(pred_logits, dim=1)
    num_classes = 3
    dice_per_class = []

    for cls in range(num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice = (2.0 * intersection + smooth) / (union + smooth)
        dice_per_class.append(dice.item())

    return dice_per_class

# Calculate per-class Dice on test set
class_dice_total = [0, 0, 0]
class_names = ['Background', 'Edema', 'Enhancing Tumor']

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        class_dice = per_class_dice(outputs, masks)
        for i in range(3):
            class_dice_total[i] += class_dice[i]

# Average
class_dice_avg = [d / len(test_loader) for d in class_dice_total]

# Print results
print("=" * 50)
print("PER-CLASS DICE SCORES")
print("=" * 50)
for i, name in enumerate(class_names):
    print(f"{name:20} | Dice: {class_dice_avg[i]:.4f}")
print("=" * 50)
print(f"Average (excluding background): {(class_dice_avg[1] + class_dice_avg[2]) / 2:.4f}")

In [ ]:
# Bar chart comparison
plt.figure(figsize=(10, 6))
bars = plt.bar(class_names, class_dice_avg, color=['gray', 'lime', 'yellow'])
plt.ylim(0, 1)
plt.ylabel('Dice Score', fontsize=12)
plt.title('Per-Class Segmentation Performance', fontsize=14)

# Add values on top of bars
for bar, value in zip(bars, class_dice_avg):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{value:.3f}', ha='center', fontsize=11)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('dice_scores.png', dpi=150)
plt.show()

In [ ]:
# Create a summary table
print("\n" + "=" * 60)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"{'Metric':<30} {'Score':<10}")
print("-" * 60)
print(f"{'Average Test Dice (Overall)':<30} {avg_test_dice:.4f}")
print(f"{'Average Test Dice (Edema)':<30} {class_dice_avg[1]:.4f}")
print(f"{'Average Test Dice (Enhancing)':<30} {class_dice_avg[2]:.4f}")
print(f"{'Average (Tumor Classes Only)':<30} {(class_dice_avg[1] + class_dice_avg[2]) / 2:.4f}")
print("=" * 60)

In [ ]:
# Define the function first
def show_sample_with_index(idx, title=""):
    """Display segmentation results for a specific test sample"""
    model.eval()
    with torch.no_grad():
        images, masks = test_dataset[idx]
        images = images.unsqueeze(0).to(device)  # Add batch dimension
        masks = masks.unsqueeze(0).to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        dice = dice_score(outputs, masks).item()

        plt.figure(figsize=(15, 5))

        # FLAIR
        plt.subplot(1, 4, 1)
        flair = images[0][3].cpu()
        flair_norm = (flair - flair.min()) / (flair.max() - flair.min() + 1e-8)
        plt.imshow(flair_norm, cmap='gray')
        plt.title(f'FLAIR (Index {idx})')
        plt.axis('off')

        # Ground Truth
        plt.subplot(1, 4, 2)
        plt.imshow(masks[0].cpu(), cmap='viridis', vmin=0, vmax=2)
        plt.title('Ground Truth')
        plt.axis('off')

        # Prediction
        plt.subplot(1, 4, 3)
        plt.imshow(preds[0].cpu(), cmap='viridis', vmin=0, vmax=2)
        plt.title(f'Prediction\nDice: {dice:.3f}')
        plt.axis('off')

        # Overlay on FLAIR
        plt.subplot(1, 4, 4)
        plt.imshow(flair_norm, cmap='gray')
        plt.imshow(preds[0].cpu(), cmap='viridis', alpha=0.5, vmin=0, vmax=2)
        plt.title('Overlay on FLAIR')
        plt.axis('off')

        if title:
            plt.suptitle(title)
        plt.tight_layout()
        plt.show()

        return dice

# Now calculate Dice for ALL test samples
print("Calculating Dice scores for all test samples...")
model.eval()
all_dice = []
all_indices = []

with torch.no_grad():
    for idx in range(len(test_dataset)):
        images, masks = test_dataset[idx]
        images = images.unsqueeze(0).to(device)
        masks = masks.unsqueeze(0).to(device)

        outputs = model(images)
        dice = dice_score(outputs, masks).item()

        all_dice.append(dice)
        all_indices.append(idx)

# Find best and worst performing samples
best_idx = all_indices[all_dice.index(max(all_dice))]
worst_idx = all_indices[all_dice.index(min(all_dice))]

# Find samples that actually have tumor (Dice > 0.01 means some tumor detected)
tumor_indices = [i for i, d in enumerate(all_dice) if d > 0.01]

if tumor_indices:
    tumor_dice = [all_dice[i] for i in tumor_indices]
    avg_tumor_dice = sum(tumor_dice) / len(tumor_dice)

    print("=" * 50)
    print("PERFORMANCE ANALYSIS")
    print("=" * 50)
    print(f"Total test samples: {len(test_dataset)}")
    print(f"Samples with tumor: {len(tumor_indices)}")
    print(f"Samples without tumor (background only): {len(test_dataset) - len(tumor_indices)}")
    print("-" * 50)
    print(f"Best Dice Score: {max(all_dice):.4f} (Sample Index {best_idx})")
    print(f"Worst Dice Score: {min(all_dice):.4f} (Sample Index {worst_idx})")
    print(f"Average Dice (all samples): {sum(all_dice)/len(all_dice):.4f}")
    print(f"Average Dice (tumor samples only): {avg_tumor_dice:.4f}")
    print("=" * 50)

    # Visualize best case
    print("\n📊 Showing BEST performing sample:")
    show_sample_with_index(best_idx, f"Best Performing Sample (Dice: {max(all_dice):.4f})")

    # Visualize worst case (if it has tumor, otherwise find worst tumor sample)
    if all_dice[worst_idx] > 0.01:
        print("\n📊 Showing WORST performing sample:")
        show_sample_with_index(worst_idx, f"Worst Performing Sample (Dice: {min(all_dice):.4f})")
    else:
        # Find worst sample that actually has tumor
        worst_tumor_idx = tumor_indices[tumor_dice.index(min(tumor_dice))]
        print(f"\n📊 Note: Overall worst sample (Index {worst_idx}) had no tumor (Dice: {all_dice[worst_idx]:.4f})")
        print(f"📊 Showing worst tumor sample instead:")
        show_sample_with_index(worst_tumor_idx, f"Worst Tumor Sample (Dice: {min(tumor_dice):.4f})")
else:
    print("No tumor samples found in test set!")